# Course 2 · Enterprise Track 0 · Preflight

**Goal:** check that course 1 left what course 2 starts from, and find out what this workspace lets you do for the rest: scheduled jobs with alerts, column masks and row filters, data-quality monitoring and system tables. You finish with a list you can paste into a request to an admin.

**Before you start:** finish course 1's enterprise track through `02_pipeline` (its last section hands the next deliveries over), and set `ENTERPRISE_CATALOG` in `enterprise/settings.py` to the catalog course 1 used.

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds course 2's helpers and checks your settings.
STEP = 0

import importlib
import sys
from pathlib import Path

def _find_track():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # course 2's enterprise/ folder in its Git folder
        if (folder / "helpers.py").exists() and (folder / "settings.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "helpers.py").exists() and (folder / "settings.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find course 2's helpers. Open this notebook from inside course 2's Git folder (enterprise/).")

TRACK = _find_track()
sys.path.insert(0, str(TRACK))
import settings
import helpers as ent
importlib.reload(settings)                                       # picks up edits to settings.py without a restart
importlib.reload(ent)
N = ent.prepare(STEP, "Preflight")

## 1. What course 1 left
Course 2 carries on with course 1's schema, its pipeline and the deliveries it handed over. It finds them by name, so `settings.py` must name the same catalog, schema and volume as course 1's settings.

In [ ]:
results = {}
try:
    spark.sql(f"DESCRIBE SCHEMA {N['fq']}").collect()
    results["schema"] = ent.check("Course 1's schema", True, N["fq"])
except Exception:
    results["schema"] = ent.check("Course 1's schema", False, f"{N['fq']} not found")
try:
    ent.find_pipeline(N)
    results["course1"] = ent.check("Course 1's pipeline", True, N["pipeline"])
except Exception as exc:
    results["course1"] = ent.check("Course 1's pipeline", False, str(exc).splitlines()[0][:160])
try:
    staged = ent.handed_over(N)["files"]
    results["handoff"] = ent.check("Handed-over deliveries", True, ", ".join(sorted(staged)))
except Exception as exc:
    results["handoff"] = ent.check("Handed-over deliveries", False, str(exc).splitlines()[0][:160])

## 2. Jobs and the Databricks SDK
Notebook 04 creates a job from Python with the Databricks SDK, the same API the UI and the CLI use, so nothing needs a terminal. Serverless environment versions ship different SDK versions (environment 4 has 0.49, environment 6 has 0.122); the track works on any of them, and uses the newer data-quality monitoring API in notebook 05 when the SDK is 0.68 or newer.

In [ ]:
try:
    import databricks.sdk
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    version = databricks.sdk.version.__version__
    results["sdk"] = ent.check("Databricks SDK", True, f"version {version}, signed in as {w.current_user.me().user_name}")
    results["dq_api"] = ent.check("Data-quality monitoring API", hasattr(w, "data_quality"),
                                  "" if hasattr(w, "data_quality") else
                                  "SDK older than 0.68: notebook 05 falls back to the older API. For the new one, "
                                  "set this notebook's environment version to 6 or newer in the Environment panel")
except Exception as exc:
    results["sdk"] = ent.check("Databricks SDK", False, str(exc).splitlines()[0][:160])

## 3. Alerts
Jobs send failure alerts to **notification destinations**: email, Slack, Microsoft Teams, PagerDuty, or a webhook. Only a workspace admin can create them; once created, anyone can attach them to a job. PagerDuty uses an Events API v2 integration key. Ticketing systems (ServiceNow, Jira, ...) would be webhook destinations, with something on their side that turns the webhook's JSON into a ticket. Copy the names you want into `ALERT_DESTINATIONS` in `enterprise/settings.py`.

In [ ]:
try:
    destinations = [(d.display_name, d.destination_type.value if d.destination_type else "") for d in w.notification_destinations.list()]
    results["destinations"] = ent.check("Notification destinations", bool(destinations),
                                        f"{len(destinations)} available" if destinations else "none yet; job failures will only email you")
    for name, kind in destinations:
        print(f"     {kind:12} {name}{'   <- configured' if name in settings.ALERT_DESTINATIONS else ''}")
except Exception as exc:
    results["destinations"] = ent.check("Notification destinations", False, str(exc).splitlines()[0][:160])

## 4. Groups and system tables
Column masks and row filters decide what to show by **account group** membership. Set `RESTRICTED_GROUP` to the group allowed to see race and religion, and `ANALYST_GROUP` to the group that gets the published view.

**System tables** (`system.access`, `system.lakeflow`, `system.data_quality_monitoring`) are the workspace's own records: lineage, job runs, audit events, quality results. An account admin enables them and grants access.

In [ ]:
for label, group in [("RESTRICTED_GROUP", settings.RESTRICTED_GROUP), ("ANALYST_GROUP", settings.ANALYST_GROUP)]:
    if not group:
        ent.check(label, False, "not set; masks will hide race and religion from everyone, which is safe")
        continue
    member = spark.sql(f"SELECT is_account_group_member('{group}')").first()[0]
    ent.check(label, True, f"{group} (you are {'a member' if member else 'not a member'})")

for table in ["system.access.table_lineage", "system.lakeflow.job_run_timeline", "system.access.audit",
              "system.data_quality_monitoring.table_results"]:
    try:
        spark.sql(f"SELECT 1 FROM {table} LIMIT 1").collect()
        results[table] = ent.check(table, True, "readable")
    except Exception:
        results[table] = ent.check(table, False, "no access (ask an admin to enable the schema and grant SELECT)")

## What to ask for
Everything marked ⚠️ above, in one list you can paste into a request. Serverless compute for notebooks, jobs and pipelines is the simplest setup; if it isn't enabled, set `ENTERPRISE_SERVERLESS = False` and set `ENTERPRISE_CLUSTER_ID` in `enterprise/settings.py` for the job's notebook task.

In [ ]:
asks = {
    "course1": "Run course 1's enterprise/02_pipeline to the end, with the same catalog and schema as settings.py.",
    "schema": "The schema course 1 created (01_unity_catalog), named the same way in settings.py.",
    "handoff": "Run section 5 of course 1's 02_pipeline: it stages the deliveries course 2 lands.",
    "sdk": "The databricks-sdk Python package on this compute.",
    "destinations": "Notification destinations for job alerts: PagerDuty (Events API v2 key) and/or a webhook for ServiceNow.",
    "system.access.table_lineage": "SELECT on system.access (lineage).",
    "system.lakeflow.job_run_timeline": "SELECT on system.lakeflow (job and pipeline history).",
    "system.access.audit": "SELECT on system.access.audit (optional: who read what).",
    "system.data_quality_monitoring.table_results": "Anomaly detection results (optional).",
}
missing = [asks[k] for k, ok in results.items() if not ok and k in asks]
print("Nothing to ask for. Go to 02b_incremental." if not missing else "Ask your workspace admin for:\n- " + "\n- ".join(missing))